# Stance classification

The stance module on the user turns against the manual annotation and the VAA answers (Section 5, Stance Classification on User Turns), and the smartvote candidate-comment benchmark (Section 3).


## 0. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

BASE_DIR = (Path(".").resolve() / "../data").resolve()
DB_DIR, FINAL_DIR = BASE_DIR / "database", BASE_DIR / "final"


def read(path):
    """As text throughout: ids are hex strings pandas would mangle, and an empty stance has
    to stay distinguishable from the string 'irrelevant'."""
    return pd.read_csv(path, dtype=str, keep_default_na=False)


# the annotation's label space (annotation-app/app.js -> LABELS)
STANCE_LABELS = ["1", "2", "3", "4", "5"]
NO_STANCE_LABELS = ["0", "content_q", "meta_q", "command", "answer"]
ALL_LABELS = STANCE_LABELS + NO_STANCE_LABELS
MEANING = {"1": "strongly against", "2": "against", "3": "neutral", "4": "in favour",
           "5": "strongly in favour", "0": "no stance", "content_q": "question about content",
           "meta_q": "meta question", "command": "command to the bot",
           "answer": "answer to the bot's question"}

# the classifier's label space -- no neutral
MODEL_TO_5 = {"disagree": 1, "rather_disagree": 2, "rather_agree": 4, "agree": 5}
MODEL_NO_STANCE = "irrelevant"

# the vote: five buttons stored as 0, .25, .5, .75, 1; 0 = against, 1 = in favour
def vote_to_5(value):
    return value * 4.0 + 1.0


# the two scales. A projection returning None means the value has no class on that scale,
# so the row is dropped -- which is how both scales end up neutral-free.
BIN2 = {1: "against", 2: "against", 4: "favour", 5: "favour"}
SCALES = {"2-class": (["against", "favour"], lambda v: None if v == 3 else BIN2[v]),
          "4-class": ([1, 2, 4, 5], lambda v: None if v == 3 else v)}
ALL_SCALES = tuple(SCALES)

## 1. Manual annotation


In [2]:
export = read(FINAL_DIR / "annotations_export.csv")
assert export["message_id"].is_unique
unknown = set(export["label"]) - set(ALL_LABELS) - {""}
assert not unknown, f"label(s) outside the app's set: {sorted(unknown)}"

export["conversation"] = export["username"] + "|" + export["condition"] + "|" + export["topic"]
export["excluded"] = export["conversation_excluded"] == "1"
export["turn_index"] = export["turn_index"].astype(int)

# the reference and the prediction, on one 1-5 scale
export["manual5"] = pd.to_numeric(export["label"].where(export["label"].isin(STANCE_LABELS)),
                                  errors="coerce")
export["manual_state"] = np.where(export["label"].isin(STANCE_LABELS), "stance", "no stance")
export["model5"] = export["model_stance"].map(MODEL_TO_5)
export["model_state"] = np.select(
    [export["model_stance"] == "", export["model_stance"] == MODEL_NO_STANCE],
    ["no prediction", "no stance"], default="stance")

turns = export[~export["excluded"] & (export["label"] != "")].copy()

print(f"{len(export)} user turns in {export['conversation'].nunique()} conversations "
      f"from {export['username'].nunique()} participants")
print(f"  - {int(export['excluded'].sum()):>4} in an excluded conversation "
      f"({export.loc[export['excluded'], 'conversation'].nunique()} conversations)")
print(f"  - {int((~export['excluded'] & (export['label'] == '')).sum()):>4} left unlabelled "
      f"(too ambiguous to place)")
print(f"  = {len(turns):>4} turns analysed, in {turns['conversation'].nunique()} conversations "
      f"from {turns['username'].nunique()} participants")
print()
display(turns["label"].value_counts().reindex(ALL_LABELS, fill_value=0).rename("turns").to_frame()
        .assign(family=["stance"] * 5 + ["no stance"] * 5,
                meaning=[MEANING[l] for l in ALL_LABELS],
                share=lambda d: (d["turns"] / len(turns)).round(3))
        .rename_axis(index="label")[["family", "meaning", "turns", "share"]])

1098 user turns in 229 conversations from 67 participants
  -  226 in an excluded conversation (38 conversations)
  -    9 left unlabelled (too ambiguous to place)
  =  863 turns analysed, in 191 conversations from 60 participants



,family,meaning,turns,share
label,,,,
1,stance,strongly against,43,0.050
2,stance,against,46,0.053
3,stance,neutral,15,0.017
4,stance,in favour,49,0.057
5,stance,strongly in favour,20,0.023
0,no stance,no stance,33,0.038
content_q,no stance,question about content,370,0.429
meta_q,no stance,meta question,29,0.034
command,no stance,command to the bot,32,0.037


In [3]:
unlabelled = export[~export["excluded"] & (export["label"] == "")]
display(unlabelled.assign(classifier=lambda d: d["model_stance"].replace("", "(no prediction)"),
                          message=lambda d: d["content"].str.slice(0, 64))
        [["message_id", "topic", "classifier", "message"]]
        .sort_values(["topic", "message_id"]).set_index("message_id"))

,topic,classifier,message
message_id,,,
1902,Education,irrelevant,EU
1907,Education,irrelevant,Ich bin gegen EU Beitritt. Weil es die Schweiz...
1909,Education,irrelevant,Ok weiter mit nächster Frage
2099,Education,irrelevant,?
1674,Energy & transport,(no prediction),Dp
1676,Energy & transport,irrelevant,Sp
1931,Foreign trade & foreign policy,irrelevant,Ja absolut
1933,Foreign trade & foreign policy,rather_agree,Ich bin klar dafür wenn die Kosten nicht explo...
1935,Foreign trade & foreign policy,irrelevant,Beenden der Umfrage


## 2. VAA answers


In [4]:
user, question = read(DB_DIR / "user.csv"), read(DB_DIR / "question.csv")
survey, raw_votes = read(DB_DIR / "survey.csv"), read(DB_DIR / "user_vote.csv")

votes = raw_votes.assign(
    username=lambda d: d["createdById"].map(user.set_index("id")["username"]),
    condition=lambda d: d["surveyId"].map(survey.set_index("id")["name"]),
    questionIndex=lambda d: d["questionId"].map(question.set_index("id")["index"]),
    vote5=lambda d: vote_to_5(pd.to_numeric(d["value"])))
chat_votes = votes[votes["questionIndex"].isin(export["questionIndex"].unique())
                   & votes["condition"].isin(["Assistant", "Interview"])].copy()

assert chat_votes["vote5"].isin([1, 2, 3, 4, 5]).all()
assert not chat_votes.duplicated(["username", "condition", "questionIndex"]).any()

print(f"{len(raw_votes)} votes -> {len(chat_votes)} on the four chat statements "
      f"({chat_votes['username'].nunique()} participants)")
print(chat_votes["vote5"].value_counts().sort_index().rename("votes").to_frame().T.to_string())

213 votes -> 213 on the four chat statements (60 participants)
vote5  1.0  2.0  3.0  4.0  5.0
votes   72   30   24   30   57


## 3. Detection of stance-bearing turns


In [5]:
print("the annotation against the classifier's raw output")
display(pd.crosstab(
    pd.Categorical(turns["label"], categories=ALL_LABELS),
    pd.Categorical(turns["model_stance"].replace("", "(no prediction)"),
                   categories=list(MODEL_TO_5) + [MODEL_NO_STANCE, "(no prediction)"]),
    dropna=False, margins=True).rename_axis(index="annotation", columns="classifier"))

the annotation against the classifier's raw output


classifier,disagree,rather_disagree,rather_agree,agree,irrelevant,(no prediction),All
annotation,,,,,,,
1,30,2,2,5,0,4,43
2,18,18,5,2,1,2,46
3,1,7,5,1,1,0,15
4,1,5,20,19,2,2,49
5,1,0,2,16,0,1,20
0,5,5,7,0,14,2,33
content_q,2,7,3,0,324,34,370
meta_q,0,0,0,0,10,19,29
command,1,1,0,0,25,5,32


In [6]:
# The binary decision, on the turns the module actually ruled on.
ruled = turns[turns["model_state"] != "no prediction"]
t = ruled["manual_state"] == "stance"
p = ruled["model_state"] == "stance"
tp, fp = int((t & p).sum()), int((~t & p).sum())
fn, tn = int((t & ~p).sum()), int((~t & ~p).sum())

rows = []
for name, (hit, miss_p, miss_r) in {"stance": (tp, fp, fn), "irrelevant": (tn, fn, fp)}.items():
    prec, rec = hit / (hit + miss_p), hit / (hit + miss_r)
    rows.append({"class": name, "n": hit + miss_r, "precision": prec, "recall": rec,
                 "f1": 2 * prec * rec / (prec + rec)})

print(f"relevance decision, n = {len(ruled)} "
      f"({int((turns['model_state'] == 'no prediction').sum())} turns had no prediction, §5)")
display(pd.crosstab(ruled["manual_state"], ruled["model_state"], margins=True)
          .rename_axis(index="annotation", columns="classifier"))
display(pd.DataFrame(rows).set_index("class").round(3))

po = (tp + tn) / len(ruled)
pe = ((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)) / len(ruled) ** 2
print(f"accuracy {po:.3f}   Cohen's kappa {(po - pe) / (1 - pe):.3f}")
print(f"share of turns that state no position: {(turns['manual_state'] == 'no stance').mean():.3f}")

relevance decision, n = 769 (94 turns had no prediction, §5)


classifier,no stance,stance,All
annotation,,,
no stance,554,51,605
stance,4,160,164
All,558,211,769


,n,precision,recall,f1
class,,,,
stance,164,0.758,0.976,0.853
irrelevant,605,0.993,0.916,0.953


accuracy 0.928   Cohen's kappa 0.807
share of turns that state no position: 0.800


In [7]:
no_stance = turns[turns["manual_state"] == "no stance"]
t_ = (pd.crosstab(no_stance["label"], no_stance["model_state"])
      .reindex(index=NO_STANCE_LABELS, columns=["no stance", "stance", "no prediction"],
               fill_value=0))
t_.insert(0, "meaning", [MEANING[l] for l in t_.index])
t_["n"] = t_[["no stance", "stance", "no prediction"]].sum(axis=1)
ruled_n = t_["no stance"] + t_["stance"]
t_["claimed a stance"] = (t_["stance"] / ruled_n.where(ruled_n > 0)).round(3)
display(t_.rename_axis(index="annotation", columns=None)
        [["meaning", "n", "no stance", "stance", "no prediction", "claimed a stance"]])
print("`claimed a stance` is a share of the turns the module ruled on, excluding `no prediction`.")

,meaning,n,no stance,stance,no prediction,claimed a stance
annotation,,,,,,
0,no stance,33,14,17,2,0.548
content_q,question about content,370,324,12,34,0.036
meta_q,meta question,29,10,0,19,0.000
command,command to the bot,32,25,2,5,0.074
answer,answer to the bot's question,226,181,20,25,0.100


`claimed a stance` is a share of the turns the module ruled on, excluding `no prediction`.


In [8]:
# Illustrative, not exhaustive. Prefix-matched so the cell degrades quietly if the export changes.
EXAMPLES = {
    "kann ich nachvollziehen": "agrees with the bot's explanation",
    "Daher bin ich mit diesen widersprüchlichen Positionen": "rejects the parties' positions",
    "ich bin aber der Meinung, dass das nicht so klar ist": "disputes what the bot said about a party",
    "Bevölkerung jeweils über die einzelnen Gesetze": "a view on direct democracy",
    "Hoffen wir das Beste.": "no position at all",
}
claimed = no_stance[no_stance["model_state"] == "stance"]
display(pd.DataFrame([
    {"message": r["content"][:52], "annotation": r["label"],
     "classifier": r["model_stance"], "what it is actually about": why}
    for needle, why in EXAMPLES.items()
    for _, r in claimed[claimed["content"].str.startswith(needle)].iterrows()
]).set_index("message"))
print(f"{len(claimed)} no-stance turns were assigned a stance in total.")

,annotation,classifier,what it is actually about
message,,,
kann ich nachvollziehen,0,rather_agree,agrees with the bot's explanation
Daher bin ich mit diesen widersprüchlichen Positione,0,rather_agree,rejects the parties' positions
"ich bin aber der Meinung, dass das nicht so klar ist",0,rather_disagree,disputes what the bot said about a party
Bevölkerung jeweils über die einzelnen Gesetze absti,0,disagree,a view on direct democracy
Hoffen wir das Beste.,0,rather_disagree,no position at all


51 no-stance turns were assigned a stance in total.


## 4. Agreement on the stance


In [9]:
def confusion(y_true, y_pred, classes, rows="annotation", cols="classifier"):
    m = pd.DataFrame(0, index=pd.Index(classes, name=rows),
                     columns=pd.Index(classes, name=cols), dtype=int)
    for a, b in zip(y_true, y_pred):
        if a in m.index and b in m.columns:
            m.loc[a, b] += 1
    return m


def per_class(cm):
    a = cm.to_numpy().astype(float)
    tp, support, predicted = np.diag(a), a.sum(1), a.sum(0)
    prec = np.divide(tp, predicted, out=np.full_like(tp, np.nan), where=predicted > 0)
    rec = np.divide(tp, support, out=np.full_like(tp, np.nan), where=support > 0)
    p0, r0 = np.nan_to_num(prec), np.nan_to_num(rec)
    f1 = np.where(p0 + r0 > 0, 2 * p0 * r0 / np.where(p0 + r0 == 0, 1, p0 + r0), 0.0)
    return pd.DataFrame({"n": support.astype(int), "predicted": predicted.astype(int),
                         "precision": prec, "recall": rec, "f1": f1}, index=cm.index).round(3)


def kappa(cm):
    a = cm.to_numpy().astype(float)
    n = a.sum()
    po = np.diag(a).sum() / n
    pe = (a.sum(0) * a.sum(1)).sum() / n ** 2
    return (po - pe) / (1 - pe)


def compare(y_true5, y_pred5, name):
    """Score a pair of 1-5 readings on both scales. `acc` is exact agreement, `macro_f1` the
    unweighted mean F1 over the classes present, `kappa` Cohen's chance-corrected agreement."""
    a, b = pd.Series(y_true5), pd.Series(y_pred5)
    keep = a.notna() & b.notna()
    a, b = a[keep].astype(int), b[keep].astype(int)
    reports, cms = [], {}
    for scale in ALL_SCALES:
        classes, project = SCALES[scale]
        x, y = a.map(project), b.map(project)
        on = x.notna() & y.notna()
        x, y = x[on], y[on]
        if all(isinstance(c, (int, np.integer)) for c in classes):
            x, y = x.astype(int), y.astype(int)
        cm = confusion(x, y, classes)
        cms[scale] = cm
        pc = per_class(cm)
        reports.append({"comparison": f"{name} · {scale}", "n": len(x),
                        "acc": np.diag(cm.to_numpy()).sum() / len(x) if len(x) else np.nan,
                        "macro_f1": pc.loc[pc["n"] > 0, "f1"].mean(),
                        "kappa": kappa(cm) if len(x) else np.nan})
    return {"report": pd.DataFrame(reports), "cm": cms,
            "per_class": {s: per_class(c) for s, c in cms.items()}}


both = turns.dropna(subset=["manual5", "model5"])
print(f"{len(both)} turns carry a stance on both sides; "
      f"{int((both['manual5'] == 3).sum())} are annotated neutral and drop out with the scales")
stance = compare(both["manual5"], both["model5"], "classifier vs annotation")
display(stance["report"].set_index("comparison").round(3))
for s in ALL_SCALES:
    print(f"{s} confusion")
    display(stance["cm"][s])
print("4-class, per class")
display(stance["per_class"]["4-class"])

160 turns carry a stance on both sides; 14 are annotated neutral and drop out with the scales


,n,acc,macro_f1,kappa
comparison,,,,
classifier vs annotation · 2-class,146,0.856,0.855,0.711
classifier vs annotation · 4-class,146,0.575,0.567,0.441


2-class confusion


classifier,against,favour
annotation,,
against,68,14
favour,7,57


4-class confusion


classifier,1,2,4,5
annotation,,,,
1,30,2,2,5
2,18,18,5,2
4,1,5,20,19
5,1,0,2,16


4-class, per class


,n,predicted,precision,recall,f1
annotation,,,,,
1,39,50,0.600,0.769,0.674
2,43,25,0.720,0.419,0.529
4,45,29,0.690,0.444,0.541
5,19,42,0.381,0.842,0.525


## 5. Turns without a prediction


In [10]:
chat_message = read(DB_DIR / "chat_message.csv")
chat_message["id_i"] = chat_message["id"].astype(int)

following = []
for _, g in chat_message.groupby(["createdById", "questionId"], sort=False):
    rows_ = g.sort_values("id_i").to_dict("records")
    for i, r in enumerate(rows_):
        if r["role"] == "user":
            reply = next((x["content"] for x in rows_[i + 1:] if x["role"] == "assistant"), "")
            following.append({"id": r["id"], "reply": reply})

msgs = (chat_message[chat_message["role"] == "user"]
        .merge(pd.DataFrame(following), on="id", how="left"))
REFUSALS = {"refused: out of scope": "ausserhalb meines Themenbereichs",
            "refused: no voting advice": "gebe ich keine Wahlempfehlungen"}
msgs["reply_kind"] = "answered"
for kind, needle in REFUSALS.items():
    msgs.loc[msgs["reply"].str.contains(needle, regex=False), "reply_kind"] = kind
msgs["missing"] = msgs["stance"] == ""

display(msgs.groupby("reply_kind")["missing"].agg(turns="size", missing="sum")
        .assign(share_missing=lambda d: (d["missing"] / d["turns"]).round(3))
        .rename_axis(index="the assistant's reply to that turn"))
answered = msgs["reply_kind"] == "answered"
print(f"{int(msgs['missing'].sum())} of {len(msgs)} user messages have no stance stored; "
      f"{int((msgs['missing'] & ~answered).sum())} sit behind a guardrail refusal.")
print(f"The residual {int((msgs['missing'] & answered).sum())} "
      f"({msgs.loc[answered, 'missing'].mean():.1%} of answered turns) are spread over the whole "
      f"field period and are shorter than average "
      f"(median {int(msgs.loc[answered & msgs['missing'], 'content'].str.len().median())} vs "
      f"{int(msgs.loc[answered & ~msgs['missing'], 'content'].str.len().median())} characters).")

,turns,missing,share_missing
the assistant's reply to that turn,,,
answered,993,51,0.051
refused: no voting advice,16,16,1.000
refused: out of scope,89,88,0.989


155 of 1098 user messages have no stance stored; 104 sit behind a guardrail refusal.
The residual 51 (5.1% of answered turns) are spread over the whole field period and are shorter than average (median 25 vs 29 characters).


## 6. Conversation-level aggregation against the VAA answer


In [11]:
def aggregate(df, col, name):
    stance_turns = df.dropna(subset=[col]).sort_values("turn_index")
    g = stance_turns.groupby("conversation")[col]
    return pd.DataFrame({f"{name}_mean": np.floor(g.mean() + 0.5).clip(1, 5),
                         f"{name}_last": g.last()})


live = export[~export["excluded"]]
conv = (live.groupby("conversation")
        .agg(username=("username", "first"), condition=("condition", "first"),
             topic=("topic", "first"), questionIndex=("questionIndex", "first"),
             n_turns=("message_id", "size")))
conv = conv.join(aggregate(live, "manual5", "annotation")).join(aggregate(live, "model5", "classifier"))
conv = conv.merge(chat_votes[["username", "condition", "questionIndex", "vote5"]],
                  on=["username", "condition", "questionIndex"], how="left")

READINGS = [("annotation", "mean"), ("annotation", "last"),
            ("classifier", "mean"), ("classifier", "last")]

def table(df, title):
    out = []
    for reading, agg in READINGS:
        rep = compare(df["vote5"], df[f"{reading}_{agg}"], f"{reading} · {agg}")["report"]
        rep[["reading", "aggregation", "scale"]] = rep["comparison"].str.split(" · ", expand=True)
        out.append(rep)
    out = pd.concat(out, ignore_index=True)
    out["scale"] = pd.Categorical(out["scale"], categories=ALL_SCALES, ordered=True)
    print(title)
    return out.set_index(["scale", "reading", "aggregation"])[["n", "acc", "macro_f1", "kappa"]].sort_index().round(3)


with_vote = conv.dropna(subset=["vote5"])
print(f"{len(conv)} conversations, {len(with_vote)} of them with a vote")

# Direction guard. `value = 1` is taken to mean "in favour", so the module's mean stance has
# to rise with the vote; if vote_to_5 were inverted every number below would silently flip
# sign and still look plausible. Checked against the classifier because it covers the most
# conversations.
check = with_vote.dropna(subset=["classifier_mean"])
r = check["classifier_mean"].corr(check["vote5"])
print(f"direction guard: corr(classifier mean, vote) = {r:.3f} over {len(check)} conversations\n")
assert r > 0.3, "vote direction looks inverted — check vote_to_5"
display(table(with_vote, "each reading on its own coverage"))

191 conversations, 182 of them with a vote
direction guard: corr(classifier mean, vote) = 0.716 over 100 conversations

each reading on its own coverage


n    acc  macro_f1  kappa
scale   reading    aggregation                            
2-class annotation last         78  0.833     0.833  0.667
                   mean         76  0.829     0.829  0.658
        classifier last         92  0.837     0.836  0.675
                   mean         81  0.901     0.901  0.802
4-class annotation last         78  0.423     0.405  0.260
                   mean         76  0.382     0.374  0.221
        classifier last         92  0.554     0.536  0.403
                   mean         81  0.543     0.528  0.398

In [12]:
cols = ["vote5"] + [f"{r}_{a}" for r, a in READINGS]
paired = conv.dropna(subset=cols)
paired = paired[(paired[cols] != 3).all(axis=1)]
display(table(paired, f"all four readings present and none neutral — n = {len(paired)}"))
print("2-class confusions on that paired set")
for reading, agg in READINGS:
    print(f"  {reading} · {agg}")
    display(compare(paired["vote5"], paired[f"{reading}_{agg}"], "x")["cm"]["2-class"]
            .rename_axis(index="vote", columns=f"{reading} ({agg})"))

all four readings present and none neutral — n = 64


n    acc  macro_f1  kappa
scale   reading    aggregation                            
2-class annotation last         64  0.859     0.859  0.719
                   mean         64  0.859     0.859  0.719
        classifier last         64  0.922     0.922  0.844
                   mean         64  0.938     0.937  0.875
4-class annotation last         64  0.453     0.432  0.291
                   mean         64  0.422     0.415  0.263
        classifier last         64  0.609     0.562  0.462
                   mean         64  0.531     0.498  0.375

2-class confusions on that paired set
  annotation · mean


annotation (mean),against,favour
vote,,
against,28,5
favour,4,27


  annotation · last


annotation (last),against,favour
vote,,
against,28,5
favour,4,27


  classifier · mean


classifier (mean),against,favour
vote,,
against,31,2
favour,2,29


  classifier · last


classifier (last),against,favour
vote,,
against,29,4
favour,1,30


## 7. Candidate-comment benchmark


In [13]:
import json

RESULTS_DIR = (Path(".").resolve() / "../evaluation/results/stance_classification").resolve()
BENCHMARK_RUN = "20260701_Qwen_Qwen3.6-35B-A3B-FP8_01"
with open(RESULTS_DIR / BENCHMARK_RUN / "stance_report.json") as f:
    report = json.load(f)

LABEL_TO_5 = {"disagree": 1, "rather_disagree": 2, "neutral": 3, "rather_agree": 4, "agree": 5}
comments = pd.DataFrame(report["case_results"])[["case_id", "prompt_category", "expected", "actual"]]
comments["truth5"] = comments["expected"].map(LABEL_TO_5)
comments["model5"] = comments["actual"].map(MODEL_TO_5)  # irrelevant -> NaN, as for the turns in §1
assert comments["truth5"].notna().all()
assert set(comments["actual"]) <= set(MODEL_TO_5) | {MODEL_NO_STANCE}, set(comments["actual"])

neutral_categories = sorted(comments.loc[comments["truth5"] == 3, "prompt_category"].unique())
print(f"{len(comments)} candidate comments on statements from {comments['prompt_category'].nunique()} "
      f"categories; {int((comments['truth5'] == 3).sum())} candidates answered neutral, all of them "
      f"on {' and '.join(neutral_categories)} statements")
print(f"the module answered `{MODEL_NO_STANCE}` on {int(comments['model5'].isna().sum())} comments "
      f"and `neutral` on none (this run cannot)\n")


def paper_accuracy(df, subset):
    """The paper's two accuracies: exact match on the four classes, and agree-vs-disagree.
    `irrelevant` is wrong on both. 2-class is left undefined while neutral references remain."""
    t, p = df["truth5"], df["model5"]
    exact = t == p
    row = {"n": len(df), "4-class correct": int(exact.sum()), "4-class": exact.mean(),
           "2-class correct": pd.NA, "2-class": np.nan}
    if (t != 3).all():
        binary = ((t >= 4) == (p >= 4)).where(p.notna(), False)
        row.update({"2-class correct": int(binary.sum()), "2-class": binary.mean()})
    return row


subsets = {
    "all comments (the run report's headline)": comments,
    "neutral answers dropped": comments[comments["truth5"] != 3],
    "categories with a neutral option dropped (the paper)":
        comments[~comments["prompt_category"].isin(neutral_categories)],
}
display(pd.DataFrame([paper_accuracy(df, s) for s, df in subsets.items()], index=list(subsets))
        .astype({"2-class correct": "Int64"}).rename_axis(index="subset").round(3))

paper_subset = subsets["categories with a neutral option dropped (the paper)"]
benchmark = compare(paper_subset["truth5"], paper_subset["model5"], "classifier vs candidate answer")
print(f"the same {len(paper_subset)} comments scored as §4 scores the turns, "
      f"`irrelevant` dropped rather than charged:")
display(benchmark["report"].set_index("comparison").round(3))

11295 candidate comments on statements from 14 categories; 299 candidates answered neutral, all of them on Federal budget and Values statements
the module answered `irrelevant` on 480 comments and `neutral` on none (this run cannot)



,n,4-class correct,4-class,2-class correct,2-class
subset,,,,,
all comments (the run report's headline),11295,6108,0.541,<NA>,NaN
neutral answers dropped,10996,6108,0.555,8787,0.799
categories with a neutral option dropped (the paper),9627,5374,0.558,7716,0.801


the same 9627 comments scored as §4 scores the turns, `irrelevant` dropped rather than charged:


,n,acc,macro_f1,kappa
comparison,,,,
classifier vs candidate answer · 2-class,9272,0.832,0.831,0.662
classifier vs candidate answer · 4-class,9272,0.580,0.573,0.441


## 8. Summary


In [14]:
summary = pd.concat([
    benchmark["report"].assign(unit="candidate comment", compared="classifier vs candidate answer"),
    stance["report"].assign(unit="turn", compared="classifier vs annotation"),
    *[compare(with_vote["vote5"], with_vote[f"{r}_{a}"], f"{r} ({a}) vs vote")["report"]
      .assign(unit="conversation", compared=f"{r} ({a}) vs vote") for r, a in READINGS],
], ignore_index=True)
summary["scale"] = pd.Categorical(summary["comparison"].str.split(" · ").str[-1],
                                  categories=ALL_SCALES, ordered=True)
display(summary.set_index(["unit", "compared", "scale"])[["n", "acc", "macro_f1", "kappa"]]
        .sort_index().round(3))
print(f"\n{len(paper_subset)} candidate comments · {len(turns)} annotated turns · "
      f"{conv.shape[0]} conversations · {turns['username'].nunique()} participants")

n    acc  macro_f1  kappa
unit              compared                       scale                                
candidate comment classifier vs candidate answer 2-class  9272  0.832     0.831  0.662
                                                 4-class  9272  0.580     0.573  0.441
conversation      annotation (last) vs vote      2-class    78  0.833     0.833  0.667
                                                 4-class    78  0.423     0.405  0.260
                  annotation (mean) vs vote      2-class    76  0.829     0.829  0.658
                                                 4-class    76  0.382     0.374  0.221
                  classifier (last) vs vote      2-class    92  0.837     0.836  0.675
                                                 4-class    92  0.554     0.536  0.403
                  classifier (mean) vs vote      2-class    81  0.901     0.901  0.802
                                                 4-class    81  0.543     0.528  0.398
turn              classifier vs annotation       2-class   146  0.856     0.855  0.711
                                                 4-class   146  0.575     0.567  0.441


9627 candidate comments · 863 annotated turns · 191 conversations · 60 participants
